Arlen said
"loops in children edges vs loops in equality edges"
That seems smart.
There are 3 meanings of eids
1. sets of terms
2. a particular term
3. a semantic value

Associating a particular term with an eid may be considered inductively during insertion time. The canonical terms coming from the union find may be someqhat ranndom or it may be determined by careful tie breaking of term ordering. Associating term with eid is kind of like having online extraction available, but maybe not a best extraction always available.
I do have a vague sense that eids = terms / eids = eclasses has some slight changes on how you implement, but I'm not sure how. You need to prevent true loops?
f(a) = a. It really should kind of be decreasing. The weight union find? 

There is a bipartite story of eclasses and enoces like the egg diagram. This is also the story I use for "sharing of parents" in deep rewrites.
The arrow ground knuth bendix story does have child arrows and rewrite arrows.

Compaction is parametrized on a notion of equality.
So really why was my observation table version weak? I think it was an example where one could
x = Cons(0, foo(x))
y = Cons(0, foo(y))
These should be compacted even though there isn't a fully observed loop.

The arena term story will probably head in the direction of eids = terms



# Compacting Arena Terms

Given a notion of equality, we may only want to retain one copy of equal things.

This can be implemented by copying over the arena into a new one. It is cute to use a union find here. I'm union find pilled, so perhaps I am biased in not finding that confusing


In [ ]:

# maybe we don't have to assume 
def canon_map(eq : list[list[bool]]) -> list[int]:
    canon = list(range())
    for i, row in enumerate(eq):
        for j,eq in 

def compact(A : Arena, canon : ) -> Arena:
    canon = canon_map(eq)
    for node in A.data:
        


    uf = list[range(self.len(A))]
    for i, node in enumerate(A.data):
        for j in eq[i][:i]:
            if j:
                uf[i] = j
                break

Hash Consing and automata minimization


# Sets of Arena Terms

This flat equational system is a reasonable way to describe sets of terms. The function symbols are implicitly set lifted.
```
x0 := { f(x1), g(x2) }
x1 := { a }
x2 := { b }
```

```
x0 :=  [f](x1) U [g](x2)
x1 := { a }
x2 := { b }
```








# non well founded sets

In python, you can't make a set of sets.
You can make a set of frozensets


In [1]:
{set()}

TypeError: unhashable type: 'set'

In [2]:
{frozenset()}

{frozenset()}

In order to do native knot tying to make cyclic terms, you need mutation. We could probably use some backdoor hack to achieve this, but I bring it up to again point out how the arena perspective makes everything less spooky. Doing this too shallowly using host language facilities just makes the whole thing more ocnfusion

Using sets is kind of silly actually, because Id being different is not enough to guarantee the sets a extesnionally different because we are not hash consing.


Egraphs despsite using hash consing still have this extensional vs intensional equality split because we don't know when new equalities are going to show up. We can tell if two ids are equal, but thir current disequality does not mean we will not eventually determine they are equal.

It is difficult to get out of the mindset of equality being an "easy" binary yes/no question in ordinary finite problems.

Futures


Black boxing of future equalities is one model. This is kind of like oracular proofs of difficulty. A quantum black box Deutsch algorithm. A black box incoming functional argument for which you are allowed to observe no structure but evaluations.

Equivalence of SAT problems (same set of solutions) is NP-hard


If you try to internalize the universe, it runs away from you. I find that a kind of compelling analogy / demonstration of the distinction between classes and sets? Or just that a set can't contain itself. But a set can contain itself in non well founded sets.


In [ ]:
from dataclasses import dataclass, field
from typing import Self
type Id = int

@dataclass
class Univ:
    sets : list[set[Id] | None] = field(default_factory=list)
    def declare(self) -> SetRef:
            new_id = len(self.sets)
            self.sets.append(None)
            return SetRef(new_id, self)
    def define(self, ref : SetRef, *args : SetRef):
            assert self.sets[ref.id] is None
            assert all(arg.arena is self for arg in args)
            self.sets[ref.id] = {arg.id for arg in args}
    def make(self, *args : SetRef) -> SetRef:
            ref = self.declare()
            self.define(ref, *args)
            return ref

# SetRef vs raw Id is nice for operator overloading
@dataclass
class SetRef:
    id : Id
    arena : Univ

    def __or__(self, other : Self) -> Self:
            assert self.arena is other.arena
            U = self.arena
            U.make(*(U.sets[self.id] | U.sets[other.id]))

    def __and__(self, other : Self) -> Self:
            assert self.arena is other.arena
            U = self.arena
            U.make(*(U.sets[self.id] & U.sets[other.id]))
    def __eq__(self, other : Self) -> bool: ...


U = Univ()
emp = U.make()
emp | emp

one = U.make(emp)
two = U.make(emp, one)
print(U)

loop = U.declare()
U.define(loop, loop)
U


Univ(sets=[set(), set(), {0}, {0, 2}])


Univ(sets=[set(), set(), {0}, {0, 2}, {4}])

In [ ]:
type NFSets = list[set[Id] | None]
def makeset(*args : SetRef) -> SetRef:
    if len(args) == 0:
        new_id = len(arena)
        arena.append(set())
        return SetRef(new_id, arena)
    arena = args[0].arena
    assert all(arg.arena is arena for arg in args)
    new_id = len(arena)
    arena.append({arg.id for arg in args})
    return SetRef(new_id, arena)
# tc is standard von neumann succ
# https://en.wikipedia.org/wiki/Transitive_set
def tc(s : SetRef) -> SetRef:
    arena = s.arena
    elmts = set()
    todo = [s.id]
    seen = set()
    while todo:
        id = todo.pop()
        if id in seen:
            continue
        seen.add(id)
        elmts.add(id)
        for id in arena[id]:
            todo.append(id)
    return makeset(*[SetRef(id, arena) for id in elmts])

In [ ]:
from dataclasses import dataclass, field
from pprint import pprint
type Id = int

@dataclass
class NFSets():
    data : list[tuple[Id,...] | None] = field(default_factory=list)

    def makevar(self):
        self.data.append(None)
        return len(self.data) - 1
    def define(self, x : Id, body : Node):
        assert self.data[x] is None
        self.data[x] = body
    def app(self, f : str, *args : Id) -> "TermRef":
        x = self.makevar()
        self.define(x, Node(f, list(args)))
        return TermRef(x, self)

@dataclass
class TermRef:
    id : Id
    arena : Arena

    def __add__(self, other : TermRef) -> TermRef:
        assert self.arena is other.arena
        return self.arena.app("+", self.id, b.id)

A = Arena()

# offset productivity
I'm _not_ crazy.

tail is anti-productive

```
zeros := Cons(0, zeros)     # 0000...
ones  := Cons(1, ones)      # 11111...
onezeros := Cons(1, zeros)  # 10000...

ones == Cons(1, tail(ones))
ones == tail(ones)
onezeros == Cons(1, tail(onezeros))
zeros == tail(onezeros)
```

logical time vs validity time.
Digital signals travelling along.
T -> Prefix where it stays self consistent with time.

zeros := Cons(0, shift(1, zeros))

Can you do negative shifts?

[[Cons(a,t)]] 
[[shift(n, s)]] = fun t -> s[t-1]





# Two Styles of Fixed point egraph

In style 1 there are definitional edges.
A variable is either a special kind of enode or it is just a name for an empty makesetted eclass

In style 2, there are no special edge. We just note that some 


There is a distinction in whether eids refer to sets of terms or a single term and then transitively sometimes to the set of terms it is equivalent to.


Nodes can be marked a productive or not. But also nodes must be marked as antiproductive also like `tail`?

`zero := Cons(0, zero)` is valid
`count := Cons(0, map(inc, count))` is valid?
`bad := Cons(0, tail(bad))` is invalid





zeros := Cons(0, zeros) 
ones  := Cons(1, ones) 
onezeros := Cons(1, zeros) 

ones == Cons(0, tail(ones))
onezero == Cons(0, tail(onezeros))


user error.
x := cons(0, tail(x))
y := cons(0, tail(y))


Fine (?)
x == cons(0, tail(x))
y == cons(0, tail(y))


# Extraction is a process
Prefix insertion - ematching
extraction is determinization




# S



In [55]:
%%file /tmp/codata.smt2
(set-logic ALL)
(declare-codatatypes ((stream 0))
 (((cons (head Int) (tail stream)))))

(declare-const zeros stream)
(assert (= zeros (cons 0 zeros)))
;(define-fun-rec zeros () stream
;    (cons 0 zeros))

(declare-const ones stream)
(assert (= ones (cons 1 ones)))

(declare-const onezeros stream)
(assert (= onezeros (cons 1 zeros)))


(assert (= ones (cons 1 (tail ones))))
(assert (= ones (tail ones)))
(assert (= onezeros (cons 1 (tail onezeros))))
(assert (= zeros (tail onezeros)))

;(assert (not (= ones onezeros)))

(check-sat)
(get-model)


Overwriting /tmp/codata.smt2


In [56]:
! cvc5 /tmp/codata.smt2 --produce-models

sat
(
(define-fun zeros () stream (cons 0 cbv_stream_0))
(define-fun ones () stream (cons 1 cbv_stream_0))
(define-fun onezeros () stream (cons 1 (cons 0 cbv_stream_0)))
)


# Bits and Bobbles




Sets of terms
eclasses as the egraph

hash consing and memoization. Taking this move too early on the arena closes off considerations of cyclic terms. You can't really hash cons cyclic terms

bisimulation

defn = [None]  as "observable ids" like in my coegraph post / open automata

More interpretations. Interpeting into set value expressions, boolean expressions.

Guardedness. Causality as a non interference property
What is a stream?
Nat -> T is extensionally the same. But this fails to expose evidence of how the thing was computed, so the intepretation is surprisingly lossy
As an engineering discipline, we want things that are physically realizable. Physically realizable filters are causal. You can get some "acausality" at the expense of delay / latency.
https://en.wikipedia.org/wiki/Causal_filter
https://en.wikipedia.org/wiki/Causality#Theories
https://www.cl.cam.ac.uk/~nk480/frp-lics11.pdf Krishnaswami and Benton
https://bentnib.org/productive.pdf

Watermarks and timely dataflow. Time stamps that aren't just int. Real time stamps, nested time stamps, partially ordered timestamps (vector clocks)


Difference equations. Differential equations. Is it always obvious that they are well formed / causal? Yeah if they are in solved form. But if it is some differential algerbaic equation thing maybe not. 
`x[t+1] := f(x[t])`

The SR latch
Maybe my confusion here and verilog confusion and frp confusion are all the same confusion?
ASP `q :- s, not ~q.  ~q :- r, not q`
ASP for boolean equation systems


`Nat -> Option<T>` gives you some partial ordering. It allows the stream to be undefined. In using a fixed point to characterize recusrive definitions, it is the domain of partial functions.

logical time of the stream vs temporal time

term distance and ultrametrics. We can define a number 1/2^d where d is the first depth of disagreement. It is calculable if they are different. If they are equal, it is 0. The iteration operation represented by a equational system is contractive on this metric

The delivery time model. Everything comes tupled with a time it is available. Functions have to be causal and return values later than they receive them. In this manner, there is kind of a difference between waiting for something before you return you answer or just returning it.

counterfactuals

Arena Term rewriting.

Rewriting with "forwarding edges". That was kind of CF's perspective on the union find https://pypy.org/posts/2022/07/toy-optimizer.html
You can rewrite over an arena term and just return the new


Cocaml examples
Set interpretations
Linear algerba interpreation
Rational numbers
Probablistic markov

Evaluating cyclic terms as intepreters for nonterminating programs / loops. CFGs.

This works in an ordinary arena. It looks an awful lot like an egraph.
The thing is, we have not defined any notion of congruence closure or rebuilding.

We could detect bisimilar stuff.

There is the make a new node rewriting vs the make a whole new arena copy rewriting.


A flat equation system describes a `env -> env` function.
we are adding an implicit fix sometimes?
an implicit
env -> (env, T)
partial fixpoints
mixed fixedpoints?

interp_mixed
finite sets or finite groups or bools or finite lattices we could actually take pairtal functions and fixedpoint them

definitional vs non efinitional in linear algebra. full rank? invertible? A x = B y  vs x := Dx  (?)
x := Ax is definitial if (I - A) is full rank. subblocks might be definitional.
differentiate the cfix vs fix via sort? What would that be for linalg?

polynomial systems
x := x**2 + y**2
y := z

could control signal vs observation vs state signal be a useful model?

functions on streams / lists
is_finite
exists
filter

non wellfounded sets

sets of things for which equality is unclear
sat problems. propagation based equality vs full sat equality (same assignments)


def mixedinterp


data LoopyTrue = Thunk LoopyTrue | True


In [ ]:
@dataclass
class Arena:
    nodes : list[Node]
    parent : list[Id]  # "better" or "rw" or "eq" edges

    def makevar(): ...
    def define(): ...
    def rw(self, src : Id, tgt : Id): ...

Notes

popl is less benchmarky vs pldi


dagger egraph bloom e-se


productive nodes 
offset union find


dpcutive cycles

F(f) <= 


forall observations, exists iterate for which it is not bottom

Observations are scott open sets? https://en.wikipedia.org/wiki/Scott_continuity

`Nat -> Option T` doesn't map back into streams quite right because I expect an entire prefix to be filled always. So there's kind of a gap there.

`Time -> List[T]`  with prefix monotonicity 
vs  `Nat -> Option (Time, T)`


Can an asbtract stream `a` have bottoms in it or is it assumed filled out?




Productivity
Monotonicity, even strict isn't the same a productive. 
Constractiveness

y=x line is fix points. A contractive function should be below the line before the crossing and bove the 
Distance to fixed point should keep decreasing. Contractive doesn't need you to have fixed point in hand though




vs seperare eclass style

is Var a separate enode constructor
A label on an empty eclass (named eclass)


class Node:
    f : str
    args : 
    productive : bool

